# Knesset votes, ballots, bills, members and seating

Collects, from the official Website API:

| Table | Grain |
|---|---|
| `fact_votes` | one plenum vote, with its reading and official tallies |
| `fact_ballots` | one member's choice in one vote |
| `bridge_vote_bills` | one accepted vote/bill link |
| `dim_bills` | one bill referenced by a vote |
| `dim_members` | one Knesset member (1,103 historical) |
| `bridge_member_factions` | one member/faction stint, dated, per Knesset |
| `dim_governments` | one dated ministerial position |

`requests-cache` makes reruns cheap. Set both limits to `None` for the full
archive; only a full run writes the committed `prepared/` tables.

Official law PDFs are **not** collected here. They are only needed by the LLooM
categorisation workflow, and `dim_bills` carries enough to fetch them later.

In [1]:
from __future__ import annotations

import re
import time
from bisect import bisect_right
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from requests_cache import CachedSession
from urllib3.util.retry import Retry

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATASET_ROOT = ROOT / 'dataset' / 'knesset_votes'

API_BASE = 'https://www.knesset.gov.il/WebSiteApi/knessetapi'
VOTE_DATES_API = f'{API_BASE}/Votes/GetAllVotesDates'
VOTE_HEADERS_API = f'{API_BASE}/Votes/GetVotesHeaders'
VOTE_DETAIL_API = f'{API_BASE}/Votes/GetVoteDetails'
BILL_DETAIL_API = f'{API_BASE}/LegislationItem/GetLegislationBillItem'
MKS_PREVIOUS_API = f'{API_BASE}/MkLobby/GetMksPrevious'
GOVERNMENT_API = f'{API_BASE}/goverment'

# Sampling limits. VOTE_DATE_SAMPLE spreads dates EVENLY across 2003-present
# rather than taking the first N: bill-type votes were 4/6 in a 2003 sample but
# 99/100 in 2024, so a head-N sample would only ever measure the 16th Knesset.
VOTE_DATE_SAMPLE = None   # None = every official vote date (2,040 of them)
BILL_LIMIT = None       # None = every bill referenced by a bill-type vote

# A full run is the only thing allowed to write the committed tables. Derived
# from the limits so the two can never disagree.
FULL_RUN = VOTE_DATE_SAMPLE is None and BILL_LIMIT is None
OUTPUT_DIR = DATASET_ROOT / ('prepared' if FULL_RUN else 'prepared_sample')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Safe default for derived-table reruns: a cache miss fails immediately rather
# than silently repeating the multi-hour scrape. Set CACHE_ONLY=False for an
# incremental fetch, or additionally FORCE_REFRESH_API=True to replace entries.
CACHE_ONLY = True
FORCE_REFRESH_API = False
NETWORK_DELAY_SECONDS = 0.1
assert not (CACHE_ONLY and FORCE_REFRESH_API)

FIRST_VOTE_DATE = date(2003, 2, 17)  # 16th Knesset onward
GOVERNMENT_IDS = range(0, 39)        # verified: GovId 0 (1948) .. 38 all return 200
BILL_ITEM_TYPE = 2                   # LU_ItemType value meaning "this item is a bill"
SEATS = 120                          # hard ceiling on ballots in one vote

# Header KnessetId/FK_Knesset identifies the bill's originating Knesset, not
# necessarily the term in which a carried-over bill was voted on. Derive the
# vote's Knesset from the official swearing-in boundaries instead.
# ponytail: finite calendar through Knesset 25; append the next swearing-in
# date before collecting votes from Knesset 26.
KNESSET_TERMS = [
    (date(2003, 2, 17), 16), (date(2006, 4, 17), 17),
    (date(2009, 2, 24), 18), (date(2013, 2, 5), 19),
    (date(2015, 3, 31), 20), (date(2019, 4, 30), 21),
    (date(2019, 10, 3), 22), (date(2020, 3, 16), 23),
    (date(2021, 4, 6), 24), (date(2022, 11, 15), 25),
]
KNESSET_START_DATES = [start for start, _ in KNESSET_TERMS]


def knesset_on(value) -> int | None:
    index = bisect_right(KNESSET_START_DATES, pd.Timestamp(value).date()) - 1
    return KNESSET_TERMS[index][1] if index >= 0 else None


assert [knesset_on(value) for value in ('2006-04-16', '2006-04-17', '2022-11-15')] == [16, 17, 25]

# The reading (קריאה) is not a field in this API; it must be inferred from the
# free-text motion in VoteHeader.Decision. ORDER MATTERS: a vote to send a bill
# to committee 'להכנה לקריאה שניה ושלישית' is a FIRST reading, so that phrase
# must be tested before the bare 'קריאה שנייה'.
# ponytail: text heuristic over free-form Hebrew motions, validated on an
# 84-vote 16th-Knesset sample where only 17 decisions mention a reading at all.
# Upgrade path = corroborate against the bill's sessionAndDocs StepTitle for the
# matching session id. Unmapped decisions stay null and are reported.
READING_RULES = [
    ('להכנה לקריאה', 1),   # sent to committee to prepare 2nd/3rd -> this vote is the 1st reading
    ('טרומית', 0),         # preliminary
    ('קריאה שלישית', 3),
    ('קריאה שנייה', 2),
    ('קריאה שניה', 2),
    ('קריאה ראשונה', 1),
]
reading_of = lambda motion: next(
    (value for phrase, value in READING_RULES if phrase in str(motion or '')), None)
assert [reading_of(m) for m in ('להעביר את הצעת החוק לוועדה להכנה לקריאה שניה ושלישית',
                               'לקבל את הצעת החוק בקריאה שלישית',
                               'לקבל בקריאה שנייה',
                               'לדחות את ההצעה')] == [1, 3, 2, None]

# Ballots carry MkName only, and token ORDER differs between endpoints
# ('אדלשטיין יולי יואל' in a ballot vs 'יולי יואל אדלשטיין' in MkLobby). This
# order-insensitive key resolved 120/120 of 2003 ballots and 114/114 of 2024
# ballots, with 0 of 1,103 names mapping to more than one MkId.
name_key = lambda value: ' '.join(sorted(
    token for token in re.sub(r'[^\w\s]', ' ', str(value or '')).split()))
assert name_key('אדלשטיין יולי יואל') == name_key('יולי יואל אדלשטיין')
assert name_key('בן-אליעזר בנימין (פואד)') == name_key('בנימין פואד בן אליעזר')

http = CachedSession(
    cache_name=DATASET_ROOT / '.http_cache',
    backend='sqlite',
    expire_after=-1,
    allowable_methods=('GET', 'POST'),
    allowable_codes=(200,),
)
http.headers.update({'User-Agent': 'lawsofisrael-data-collection/0.2', 'Accept': 'application/json'})
http.mount('https://', HTTPAdapter(max_retries=Retry(
    total=4, backoff_factor=0.5, status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods={'GET', 'POST'})))

print('Full run:', FULL_RUN, '-> writes', OUTPUT_DIR.relative_to(ROOT))
print('Limits:', {'vote_date_sample': VOTE_DATE_SAMPLE, 'bills': BILL_LIMIT})
print('HTTP mode:', 'cache only' if CACHE_ONLY else ('force refresh' if FORCE_REFRESH_API else 'cache + fetch misses'))

Full run: True -> writes dataset/knesset_votes/prepared
Limits: {'vote_date_sample': None, 'bills': None}
HTTP mode: cache only


## Request helpers

The vote routes return legacy `System.Data.DataSet` objects, so responses need
flattening before use. Both helpers are used by several cells below.

In [2]:
def get_json(url: str, *, params: dict | None = None, payload: dict | None = None) -> dict | list:
    options = {'timeout': 60, 'only_if_cached': CACHE_ONLY, 'force_refresh': FORCE_REFRESH_API}
    response = (http.get(url, params=params, **options) if payload is None
                else http.post(url, json=payload, **options))
    if CACHE_ONLY and not getattr(response, 'from_cache', False):
        raise RuntimeError(f'Cache miss for {response.request.url}; set CACHE_ONLY=False to fetch')
    response.raise_for_status()
    if 'json' not in response.headers.get('Content-Type', '').lower() or not response.content:
        raise ValueError(f'Expected non-empty JSON from {response.url}')
    if not getattr(response, 'from_cache', False):
        time.sleep(NETWORK_DELAY_SECONDS)
    return response.json()


def dataset_rows(payload: dict) -> list[dict]:
    tables = payload.get('Tables', payload.get('tables', payload))
    if isinstance(tables, dict):
        return [row for table in tables.values() if isinstance(table, list) for row in table if isinstance(row, dict)]
    if isinstance(tables, list):
        return [row for row in tables if isinstance(row, dict)]
    return []


assert dataset_rows({'Table': [{'VoteId': 1}]}) == [{'VoteId': 1}]

## 1. Discover dates and collect vote headers

One request per official vote date. There is no by-session vote endpoint, so the
date axis is the only complete enumerator of votes.

In [3]:
all_dates = sorted({pd.Timestamp(row['start']).date()
                    for row in dataset_rows(get_json(VOTE_DATES_API)) if row.get('start')})
in_scope_dates = [value for value in all_dates if value >= FIRST_VOTE_DATE]
selected_dates = in_scope_dates
if VOTE_DATE_SAMPLE:
    stride = max(1, len(in_scope_dates) // VOTE_DATE_SAMPLE)
    selected_dates = in_scope_dates[::stride][:VOTE_DATE_SAMPLE]
print(f'Vote dates: {len(selected_dates):,} of {len(in_scope_dates):,} in scope')
print(f'  span: {selected_dates[0]} .. {selected_dates[-1]}')

header_rows, header_status = [], []
for index, vote_date in enumerate(selected_dates, 1):
    try:
        rows = dataset_rows(get_json(VOTE_HEADERS_API, payload={
            'SearchType': 1, 'FromDate': vote_date.isoformat(), 'ToDate': vote_date.isoformat()}))
        header_rows.extend({**row, 'discovery_date': vote_date.isoformat()} for row in rows)
        header_status.append({'vote_date': vote_date.isoformat(), 'state': 'available', 'error': None})
    except (requests.RequestException, ValueError) as error:
        header_status.append({'vote_date': vote_date.isoformat(), 'state': 'fetch_failed', 'error': str(error)})
    if index == 1 or index % 25 == 0 or index == len(selected_dates):
        print(f'  headers {index:,}/{len(selected_dates):,} ({vote_date.isoformat()})')

headers_df = pd.DataFrame(header_rows)
header_status_df = pd.DataFrame(header_status, columns=['vote_date', 'state', 'error'])
print(f'Indexed votes: {len(headers_df):,}; date failures: {int(header_status_df["state"].ne("available").sum())}')

Vote dates: 2,040 of 2,040 in scope
  span: 2003-03-26 .. 2026-07-28
  headers 1/2,040 (2003-03-26)
  headers 25/2,040 (2003-12-15)
  headers 50/2,040 (2004-02-17)
  headers 75/2,040 (2004-05-19)
  headers 100/2,040 (2004-07-21)
  headers 125/2,040 (2004-12-07)
  headers 150/2,040 (2005-02-07)


  headers 175/2,040 (2005-04-20)
  headers 200/2,040 (2005-07-18)


  headers 225/2,040 (2005-12-05)
  headers 250/2,040 (2006-06-27)
  headers 275/2,040 (2006-11-01)
  headers 300/2,040 (2007-01-01)
  headers 325/2,040 (2007-02-27)
  headers 350/2,040 (2007-06-20)
  headers 375/2,040 (2007-10-24)
  headers 400/2,040 (2007-12-26)
  headers 425/2,040 (2008-02-20)


  headers 450/2,040 (2008-05-27)
  headers 475/2,040 (2008-07-28)


  headers 500/2,040 (2009-05-05)
  headers 525/2,040 (2009-07-06)
  headers 550/2,040 (2009-11-11)
  headers 575/2,040 (2010-01-11)
  headers 600/2,040 (2010-03-15)
  headers 625/2,040 (2010-06-21)
  headers 650/2,040 (2010-11-02)
  headers 675/2,040 (2011-01-04)
  headers 700/2,040 (2011-03-07)
  headers 725/2,040 (2011-06-15)
  headers 750/2,040 (2011-10-31)


...

  headers 1,475/2,040 (2020-09-23)
  headers 1,500/2,040 (2020-12-14)
  headers 1,525/2,040 (2021-05-05)
  headers 1,550/2,040 (2021-07-15)
  headers 1,575/2,040 (2021-11-05)
  headers 1,600/2,040 (2022-01-04)
  headers 1,625/2,040 (2022-03-07)
  headers 1,650/2,040 (2022-06-20)
  headers 1,675/2,040 (2023-01-16)
  headers 1,700/2,040 (2023-03-20)


  headers 1,725/2,040 (2023-06-19)
  headers 1,750/2,040 (2023-10-25)
  headers 1,775/2,040 (2024-01-15)


  headers 1,800/2,040 (2024-03-27)
  headers 1,825/2,040 (2024-07-08)
  headers 1,850/2,040 (2024-11-08)
  headers 1,875/2,040 (2025-01-06)
  headers 1,900/2,040 (2025-03-04)
  headers 1,925/2,040 (2025-05-28)
  headers 1,950/2,040 (2025-10-20)
  headers 1,975/2,040 (2025-12-24)
  headers 2,000/2,040 (2026-03-11)
  headers 2,025/2,040 (2026-06-11)


  headers 2,040/2,040 (2026-07-28)


Indexed votes: 36,074; date failures: 0


## 2. Collect vote details, ballots and bill references

Each official vote ID is fetched once. `VoteHeader.FK_ItemID` is the bill ID
**only** when `LU_ItemType == 2`; measured 133/133 such items resolved against
the bill endpoint, echoing the same `Id` with an exact title match, while 0/20
non-type-2 items resolved. Agenda-item IDs share the bill ID number space
(16403 is a bill, 16060 is an agenda item), so the type filter is what prevents
fabricated links.

In [4]:
vote_ids = sorted({int(v) for v in headers_df['VoteId'].dropna()}) if not headers_df.empty else []
vote_details, detail_status = {}, []
for index, vote_id in enumerate(vote_ids, 1):
    try:
        vote_details[vote_id] = get_json(f'{VOTE_DETAIL_API}/{vote_id}')
        detail_status.append({'vote_id': vote_id, 'detail_state': 'available', 'error': None})
    except (requests.RequestException, ValueError) as error:
        detail_status.append({'vote_id': vote_id, 'detail_state': 'fetch_failed', 'error': str(error)})
    if index == 1 or index % 100 == 0 or index == len(vote_ids):
        print(f'  details {index:,}/{len(vote_ids):,}')
detail_status_df = pd.DataFrame(detail_status, columns=['vote_id', 'detail_state', 'error'])

vote_item_links = []
for vote_id, payload in vote_details.items():
    header = (payload.get('VoteHeader') or [{}])[0]
    if header.get('FK_ItemID') is not None:
        vote_item_links.append({'vote_id': vote_id, 'bill_id': int(header['FK_ItemID']),
                                'lu_item_type': header.get('LU_ItemType'),
                                'decision': header.get('Decision')})
bill_links = [link for link in vote_item_links if link['lu_item_type'] == BILL_ITEM_TYPE]
print(f'Details: {len(vote_details):,}/{len(vote_ids):,}; item refs: {len(vote_item_links):,}; '
      f'of bill type: {len(bill_links):,}')

  details 1/36,054
  details 100/36,054
  details 200/36,054


  details 300/36,054


  details 400/36,054
  details 500/36,054
  details 600/36,054


  details 700/36,054


  details 800/36,054
  details 900/36,054
  details 1,000/36,054


...

  details 34,700/36,054
  details 34,800/36,054
  details 34,900/36,054
  details 35,000/36,054


  details 35,100/36,054
  details 35,200/36,054
  details 35,300/36,054


  details 35,400/36,054
  details 35,500/36,054
  details 35,600/36,054
  details 35,700/36,054


  details 35,800/36,054
  details 35,900/36,054
  details 36,000/36,054
  details 36,054/36,054


Details: 36,054/36,054; item refs: 36,054; of bill type: 27,900


## 3. Collect members and seating history

`MkLobby/GetMksPrevious` is a single request returning `factionPrevious` (4,487
rows of MkId + Knesset + faction + `beginDate`, covering all 1,103 historical
MKs) and `replacement` (378 mid-term replacements). Faction changes *within* a
Knesset are preserved as separate stints.

`goverment?GovId=N` adds dated ministerial positions for governments 0-38, back
to 1948. It identifies people by `FK_SanID` and `MkName` rather than `MkId`, so
positions join to members through the same order-insensitive name key.

In [5]:
members_payload = get_json(MKS_PREVIOUS_API, params={'lang': 'he'})
faction_rows = [{'member_id': row.get('MkId'), 'member_name': row.get('FullName'),
                 'knesset_num': row.get('KnessetId'), 'faction_name': row.get('FactionName'),
                 'begin_date': row.get('beginDate'), 'max_knesset_num': row.get('MaxKnessetId')}
                for row in members_payload.get('factionPrevious') or []]
print(f"Faction stints: {len(faction_rows):,}; expected members: {members_payload.get('CountPreviousMks')}")

government_rows = []
for gov_id in GOVERNMENT_IDS:
    try:
        payload = get_json(GOVERNMENT_API, params={'GovId': gov_id, 'Lang': 'HE'})
    except (requests.RequestException, ValueError) as error:
        print(f'  government {gov_id} failed: {error}')
        continue
    government_rows.extend({
        'government_id': position.get('GovermentId'), 'government_name': position.get('GovermentName'),
        'position_name': position.get('PositionName'), 'ministry_name': position.get('MinistryName'),
        'member_name_source': position.get('MkName'), 'source_person_id': position.get('FK_SanID'),
        'faction_name': position.get('FactionName'), 'source_faction_id': position.get('faction_id'),
        'knesset_num': position.get('knesset'), 'is_mk': position.get('IsMK'),
        'start_date': position.get('PositionStratDate'), 'finish_date': position.get('PositionFinishedDate'),
    } for position in payload.get('GovermentPositions') or [])
print(f'Government positions: {len(government_rows):,}')

Faction stints: 4,487; expected members: 1103
Government positions: 2,546


## 4. Collect the referenced bills

The universe is exactly the bills named by bill-type votes. Bills that were
discussed but never voted are deliberately out of scope.

In [6]:
candidate_bill_ids = sorted({link['bill_id'] for link in bill_links})
selected_bill_ids = candidate_bill_ids if BILL_LIMIT is None else candidate_bill_ids[:BILL_LIMIT]
bill_details, bill_status = {}, []
for index, bill_id in enumerate(selected_bill_ids, 1):
    try:
        bill_details[bill_id] = get_json(BILL_DETAIL_API, params={'ItemId': bill_id})
        bill_status.append({'bill_id': bill_id, 'state': 'available', 'error': None})
    except (requests.RequestException, ValueError) as error:
        bill_status.append({'bill_id': bill_id, 'state': 'fetch_failed', 'error': str(error)})
    if index == 1 or index % 100 == 0 or index == len(selected_bill_ids):
        print(f'  bills {index:,}/{len(selected_bill_ids):,}')
bill_status_df = pd.DataFrame(bill_status, columns=['bill_id', 'state', 'error'])
print(f'Bill universe: {len(candidate_bill_ids):,}; collected: {len(bill_details):,}')

  bills 1/8,317
  bills 100/8,317


  bills 200/8,317


  bills 300/8,317


  bills 400/8,317
  bills 500/8,317


  bills 600/8,317
  bills 700/8,317
  bills 800/8,317
  bills 900/8,317


...

  bills 6,700/8,317
  bills 6,800/8,317
  bills 6,900/8,317
  bills 7,000/8,317


  bills 7,100/8,317
  bills 7,200/8,317
  bills 7,300/8,317


  bills 7,400/8,317
  bills 7,500/8,317
  bills 7,600/8,317
  bills 7,700/8,317


  bills 7,800/8,317
  bills 7,900/8,317
  bills 8,000/8,317
  bills 8,100/8,317


  bills 8,200/8,317
  bills 8,300/8,317
  bills 8,317/8,317
Bill universe: 8,317; collected: 8,317


## 5. Build the tables

Dimensions that would be pure projections of `fact_votes` (knessets, sessions,
agenda items) are deliberately not materialised.

In [7]:
# Preserve every discovery date per vote instead of collapsing to one row.
header_rows_by_vote = {}
for row in (headers_df.dropna(subset=['VoteId']).itertuples(index=False) if not headers_df.empty else []):
    header_rows_by_vote.setdefault(int(row.VoteId), []).append(row._asdict())
detail_state_by_vote = dict(zip(detail_status_df['vote_id'], detail_status_df['detail_state']))

member_by_knesset, member_by_any = {}, {}
for row in faction_rows:
    key = name_key(row['member_name'])
    if key and row['member_id'] is not None:
        member_by_any.setdefault(key, row['member_id'])
        if row['knesset_num'] is not None:
            member_by_knesset.setdefault((int(row['knesset_num']), key), row['member_id'])

fact_vote_rows, ballot_rows, counter_rows, secret_result_rows = [], [], [], []
for vote_id, header_list in header_rows_by_vote.items():
    header = header_list[0]
    detail = vote_details.get(vote_id, {})
    detail_header = (detail.get('VoteHeader') or [{}])[0]
    counters = detail.get('VoteCounters') or []
    vote_datetime = header.get('VoteDate') or detail_header.get('VoteDate') or header.get('discovery_date')
    source_knesset_num = header.get('KnessetId') or detail_header.get('FK_Knesset')
    knesset_num = knesset_on(vote_datetime)
    decision = detail_header.get('Decision')
    fact_vote_rows.append({
        'vote_id': vote_id,
        'discovery_date': header.get('discovery_date'),
        'discovery_date_count': len(header_list),
        'detail_state': detail_state_by_vote.get(vote_id, 'fetch_failed'),
        'knesset_num': knesset_num,
        # Preserve the API value for provenance. On carried-over bills it is
        # the bill's originating Knesset, so it must not label the vote itself.
        'source_knesset_num': source_knesset_num,
        'plenum_session_id': header.get('SessionId'),
        'item_id': detail_header.get('FK_ItemID'),
        'lu_item_type': detail_header.get('LU_ItemType'),
        'vote_datetime': vote_datetime,
        'vote_type': header.get('VoteType') or detail_header.get('VoteType'),
        'item_title': header.get('ItemTitle') or detail_header.get('ItemTitle'),
        'decision': decision,
        # Null means "reading not stated in the vote motion", not "no reading".
        # User-facing views should show that label and the original decision.
        'reading': reading_of(decision),
        'accepted': detail_header.get('IsForAccepted'),
        # Official aggregate, reconciled against ballot rows in the report.
        'counter_total': int(sum(c.get('countOfResult') or 0 for c in counters)) if counters else None,
    })
    for ordinal, counter in enumerate(counters):
        counter_rows.append({
            'counter_id': f'{vote_id}:{ordinal}', 'vote_id': vote_id,
            'result_label': counter.get('Title'), 'vote_count': counter.get('countOfResult'),
            'source_rank': counter.get('rn'), 'source_color': counter.get('ColorName'),
        })
    # Secret votes publish candidate/result aggregates separately and usually
    # cannot publish member ballots. Keep those official totals instead.
    for ordinal, result in enumerate(detail.get('DescreetVoteResults') or []):
        secret_result_rows.append({
            'secret_result_id': f"{vote_id}:{result.get('ID', ordinal)}",
            'vote_id': vote_id, 'source_result_id': result.get('ID'),
            'result_label': result.get('description'), 'votes_for': result.get('AmountFor'),
            'votes_against': result.get('AmountAgainst'),
            'is_single_nominee': result.get('IsSingleNominee'),
            'source_rank': result.get('rn'), 'source_color': result.get('ColorName'),
        })
    for ordinal, ballot in enumerate(detail.get('VoteDetails') or []):
        key = name_key(ballot.get('MkName'))
        member_id = member_by_knesset.get((int(knesset_num), key)) if pd.notna(knesset_num) else None
        ballot_rows.append({
            'ballot_id': f'{vote_id}:{ordinal}',
            'vote_id': vote_id,
            'member_id': member_id if member_id is not None else member_by_any.get(key),
            'member_name_source': ballot.get('MkName'),
            'source_faction_name': ballot.get('FactionName'),
            'vote_choice_raw': ballot.get('Title'),
            'vote_result_id': ballot.get('VoteResultId'),
        })

fact_votes = pd.DataFrame(fact_vote_rows, columns=[
    'vote_id', 'discovery_date', 'discovery_date_count', 'detail_state', 'knesset_num',
    'source_knesset_num', 'plenum_session_id', 'item_id', 'lu_item_type', 'vote_datetime',
    'vote_type', 'item_title', 'decision', 'reading', 'accepted', 'counter_total'])
fact_ballots = pd.DataFrame(ballot_rows, columns=[
    'ballot_id', 'vote_id', 'member_id', 'member_name_source', 'source_faction_name',
    'vote_choice_raw', 'vote_result_id'])
fact_vote_counters = pd.DataFrame(counter_rows, columns=[
    'counter_id', 'vote_id', 'result_label', 'vote_count', 'source_rank', 'source_color'])
fact_secret_vote_results = pd.DataFrame(secret_result_rows, columns=[
    'secret_result_id', 'vote_id', 'source_result_id', 'result_label', 'votes_for',
    'votes_against', 'is_single_nominee', 'source_rank', 'source_color'])

bridge_member_factions = pd.DataFrame(faction_rows, columns=[
    'member_id', 'member_name', 'knesset_num', 'faction_name', 'begin_date',
    'max_knesset_num']).drop_duplicates()
dim_members = (bridge_member_factions[['member_id', 'member_name', 'max_knesset_num']]
               .dropna(subset=['member_id']).drop_duplicates('member_id'))
dim_governments = pd.DataFrame(government_rows, columns=[
    'government_id', 'government_name', 'position_name', 'ministry_name', 'member_name_source',
    'source_person_id', 'faction_name', 'source_faction_id', 'knesset_num', 'is_mk',
    'start_date', 'finish_date'])
dim_governments['member_id'] = dim_governments['member_name_source'].map(
    lambda value: member_by_any.get(name_key(value)))

# Keep a row for bills whose detail fetch failed so the universe stays accounted for.
bill_rows = [{'bill_id': bill_id, 'detail_state': 'available',
              **{field: (bill.get('general') or {}).get(source) for field, source in (
                  ('name', 'Name'), ('status', 'Status'), ('initiators', 'Initiators'),
                  ('knesset_num', 'Knesset'), ('proposal_type', 'SubType'),
                  ('committee_name', 'CommitteeName'), ('commencement_date', 'CommencementDate'),
                  ('summary', 'SummaryLaw'), ('publication_date', 'PublicationSeriesLaw'),
                  ('previous_names', 'PreviousNameList'))}}
             for bill_id, bill in bill_details.items()]
bill_rows += [{'bill_id': row.bill_id, 'detail_state': row.state}
              for row in bill_status_df[bill_status_df['state'].ne('available')].itertuples(index=False)]
dim_bills = pd.DataFrame(bill_rows, columns=[
    'bill_id', 'detail_state', 'name', 'status', 'initiators', 'knesset_num', 'proposal_type',
    'committee_name', 'commencement_date', 'summary', 'publication_date', 'previous_names'])

# A link is accepted only when the item is of bill type AND resolves to a
# collected bill. Both are required: the ID space is shared with agenda items.
available_bill_ids = set(dim_bills.loc[dim_bills['detail_state'].eq('available'), 'bill_id'])
bridge_vote_bills = pd.DataFrame(
    [{'vote_id': link['vote_id'], 'bill_id': link['bill_id'], 'decision': link['decision'],
      'reading': reading_of(link['decision']), 'source_endpoint': VOTE_DETAIL_API,
      'source_fields': ['FK_ItemID', 'LU_ItemType']}
     for link in bill_links if link['bill_id'] in available_bill_ids],
    columns=['vote_id', 'bill_id', 'decision', 'reading', 'source_endpoint', 'source_fields'],
).drop_duplicates(['vote_id', 'bill_id'])
unconfirmed_links = [link for link in bill_links if link['bill_id'] not in available_bill_ids]

print({'fact_votes': len(fact_votes), 'fact_ballots': len(fact_ballots),
       'fact_vote_counters': len(fact_vote_counters),
       'fact_secret_vote_results': len(fact_secret_vote_results),
       'bridge_vote_bills': len(bridge_vote_bills), 'dim_bills': len(dim_bills),
       'dim_members': len(dim_members), 'faction_stints': len(bridge_member_factions),
       'gov_positions': len(dim_governments)})

{'fact_votes': 36054, 'fact_ballots': 1948412, 'fact_vote_counters': 67454, 'fact_secret_vote_results': 18, 'bridge_vote_bills': 27900, 'dim_bills': 8317, 'dim_members': 1103, 'faction_stints': 4487, 'gov_positions': 2546}


## 6. Integrity report and write

Every check is a count of offending rows against a limit, so one table covers
referential integrity, ballot/aggregate reconciliation, identity resolution,
null pressure and anomalies. `blocking` failures stop a full run; `review`
failures are published as numbers.

In [8]:
tables = {
    'fact_votes': fact_votes, 'fact_ballots': fact_ballots,
    'fact_vote_counters': fact_vote_counters,
    'fact_secret_vote_results': fact_secret_vote_results,
    'bridge_vote_bills': bridge_vote_bills, 'dim_bills': dim_bills,
    'dim_members': dim_members, 'bridge_member_factions': bridge_member_factions,
    'dim_governments': dim_governments,
}

ballots_per_vote = fact_ballots.groupby('vote_id').size()
# Only electronic/named votes promise member ballots. Reindex instead of an
# inner join so a positive official counter with zero ballots cannot disappear.
member_counter_total = (fact_votes.loc[
    fact_votes['vote_type'].isin(['אלקטרונית', 'שמית']) & fact_votes['counter_total'].notna(),
    ['vote_id', 'counter_total']].set_index('vote_id')['counter_total'])
actual_ballots = ballots_per_vote.reindex(member_counter_total.index, fill_value=0)
unreconciled = int((member_counter_total.astype(int) != actual_ballots.astype(int)).sum())
expected_counter_rows = sum(len(payload.get('VoteCounters') or []) for payload in vote_details.values())
expected_secret_results = sum(len(payload.get('DescreetVoteResults') or []) for payload in vote_details.values())

# A resolving bill ID proves the item exists; matching titles prove the link
# points at the RIGHT bill. Measured 122/122 exact matches on sample dates.
link_titles = (bridge_vote_bills[['vote_id', 'bill_id']]
               .merge(fact_votes[['vote_id', 'item_title']], on='vote_id', how='left')
               .merge(dim_bills[['bill_id', 'name']], on='bill_id', how='left'))
tidy = lambda column: (column.fillna('').astype(str)
                       .str.replace(r'\s+', ' ', regex=True).str.strip().str.rstrip('.'))
link_title_mismatch = int((tidy(link_titles['item_title']) != tidy(link_titles['name'])).sum())

required_fields = {
    'fact_votes': ['vote_id', 'knesset_num', 'vote_datetime'],
    'fact_ballots': ['vote_id', 'vote_choice_raw'],
    'fact_vote_counters': ['vote_id', 'result_label', 'vote_count'],
    'fact_secret_vote_results': ['vote_id', 'result_label', 'votes_for', 'votes_against'],
    'bridge_vote_bills': ['vote_id', 'bill_id'],
    'dim_bills': ['bill_id'],
    'dim_members': ['member_id'],
}
required_nulls = sum(int(tables[table][column].isna().sum())
                     for table, columns in required_fields.items() for column in columns)

vote_times = pd.to_datetime(fact_votes['vote_datetime'], errors='coerce')
votes_with_ballots = set(ballots_per_vote.index)
secret_vote_ids = set(fact_secret_vote_results['vote_id'])
positive_member_votes = fact_votes.loc[
    fact_votes['vote_type'].isin(['אלקטרונית', 'שמית']) & fact_votes['counter_total'].fillna(0).gt(0),
    'vote_id']
# Non-MK ministers correctly have no MkId. Government 0 is the provisional
# cabinet before the first Knesset, despite one loose IsMK=True source flag.
expected_mk_positions = (dim_governments['is_mk'].eq(True)
                         & dim_governments['knesset_num'].fillna(0).astype(int).ge(1))
source_empty_electronic = int((fact_votes['vote_type'].eq('אלקטרונית')
                               & fact_votes['counter_total'].isna()
                               & ~fact_votes['vote_id'].isin(votes_with_ballots)).sum())
source_knesset_differences = int((fact_votes['source_knesset_num'].notna()
                                  & fact_votes['source_knesset_num'].astype('Int64').ne(
                                      fact_votes['knesset_num'].astype('Int64'))).sum())

checks = [
    ('duplicate vote_id in fact_votes', int(fact_votes['vote_id'].duplicated().sum()), 0, 'blocking'),
    ('ballot vote_id absent from fact_votes', int((~fact_ballots['vote_id'].isin(fact_votes['vote_id'])).sum()), 0, 'blocking'),
    ('counter vote_id absent from fact_votes', int((~fact_vote_counters['vote_id'].isin(fact_votes['vote_id'])).sum()), 0, 'blocking'),
    ('secret-result vote_id absent from fact_votes', int((~fact_secret_vote_results['vote_id'].isin(fact_votes['vote_id'])).sum()), 0, 'blocking'),
    ('duplicate member within one vote', int(fact_ballots.duplicated(['vote_id', 'member_name_source']).sum()), 0, 'blocking'),
    ('duplicate counter_id', int(fact_vote_counters['counter_id'].duplicated().sum()), 0, 'blocking'),
    ('duplicate secret_result_id', int(fact_secret_vote_results['secret_result_id'].duplicated().sum()), 0, 'blocking'),
    (f'ballots in one vote above {SEATS} seats', int((ballots_per_vote > SEATS).sum()), 0, 'blocking'),
    ('bridge bill_id absent from dim_bills', int((~bridge_vote_bills['bill_id'].isin(dim_bills['bill_id'])).sum()), 0, 'blocking'),
    ('bridge vote_id absent from fact_votes', int((~bridge_vote_bills['vote_id'].isin(fact_votes['vote_id'])).sum()), 0, 'blocking'),
    ('duplicate (vote_id, bill_id) in bridge', int(bridge_vote_bills.duplicated(['vote_id', 'bill_id']).sum()), 0, 'blocking'),
    ('counter rows dropped during normalization', abs(expected_counter_rows - len(fact_vote_counters)), 0, 'blocking'),
    ('secret results dropped during normalization', abs(expected_secret_results - len(fact_secret_vote_results)), 0, 'blocking'),
    ('member ballots != VoteCounters total', unreconciled, 0, 'blocking'),
    # MkLobby resolved 120/120 of 2003 ballots and 114/114 of 2024 ballots, so an
    # unresolved ballot is a real defect rather than a gap in the roster.
    ('ballots with no official member_id', int(fact_ballots['member_id'].isna().sum()), 0, 'blocking'),
    ('faction stints with no member_id', int(bridge_member_factions['member_id'].isna().sum()), 0, 'blocking'),
    ('nulls in required fields', required_nulls, 0, 'blocking'),
    ('header dates that failed to fetch', int(header_status_df['state'].ne('available').sum()), 0, 'blocking'),
    ('member names mapping to several IDs', int((dim_members.assign(
        key=dim_members['member_name'].map(name_key)).groupby('key')['member_id'].nunique() > 1).sum()), 0, 'blocking'),
    ('bridge rows where bill name != vote title', link_title_mismatch, 0, 'review'),
    ('bill-type item ids not confirmed as bills', len(unconfirmed_links), 0, 'review'),
    ('Knesset-member government positions unmatched', int((expected_mk_positions
        & dim_governments['member_id'].isna()).sum()), 0, 'review'),
    ('member-level votes with counters but zero ballots', int((~positive_member_votes.isin(votes_with_ballots)).sum()), 0, 'review'),
    ('secret votes with no aggregate results', int((fact_votes['vote_type'].eq('חשאית')
        & ~fact_votes['vote_id'].isin(secret_vote_ids)).sum()), 0, 'review'),
    ('vote_datetime outside 2003..today', int(((vote_times < pd.Timestamp(FIRST_VOTE_DATE))
                                              | (vote_times > pd.Timestamp.now())).sum()), 0, 'review'),
    ('vote Knesset disagrees with term calendar', int((fact_votes['knesset_num'].astype('Int64')
        != vote_times.map(knesset_on).astype('Int64')).sum()), 0, 'blocking'),
]
quality_report = pd.DataFrame(checks, columns=['check', 'violations', 'limit', 'severity'])
quality_report['status'] = np.where(quality_report['violations'] <= quality_report['limit'], 'pass', 'fail')

# Self-check on the status rule itself.
_demo = pd.DataFrame([('a', 0, 0), ('b', 3, 0), ('c', 2, 5)], columns=['check', 'violations', 'limit'])
assert list(np.where(_demo['violations'] <= _demo['limit'], 'pass', 'fail')) == ['pass', 'fail', 'pass']

linked_votes = set(bridge_vote_bills['vote_id'])
coverage = fact_votes.assign(linked=fact_votes['vote_id'].isin(linked_votes)).groupby(
    ['knesset_num', 'lu_item_type'])['linked'].agg(['size', 'sum'])
coverage['linked_pct'] = (100 * coverage['sum'] / coverage['size']).round(1)

collection_summary = pd.DataFrame([
    {'metric': 'full_run', 'count': int(FULL_RUN)},
    {'metric': 'vote_dates', 'count': len(header_status_df)},
    {'metric': 'votes', 'count': len(fact_votes)},
    {'metric': 'ballots', 'count': len(fact_ballots)},
    {'metric': 'vote_counter_rows', 'count': len(fact_vote_counters)},
    {'metric': 'secret_vote_results', 'count': len(fact_secret_vote_results)},
    {'metric': 'vote_bill_links', 'count': len(bridge_vote_bills)},
    {'metric': 'bill_votes_reading_not_stated', 'count': int(bridge_vote_bills['reading'].isna().sum())},
    {'metric': 'source_empty_electronic_votes', 'count': source_empty_electronic},
    {'metric': 'source_knesset_differs_from_vote', 'count': source_knesset_differences},
    {'metric': 'bills', 'count': len(dim_bills)},
    {'metric': 'members', 'count': len(dim_members)},
    {'metric': 'faction_stints', 'count': len(bridge_member_factions)},
    {'metric': 'government_positions', 'count': len(dim_governments)},
])

failures = quality_report[quality_report['status'].eq('fail')]
print(f'Quality: {len(quality_report) - len(failures)}/{len(quality_report)} checks pass')
display(failures if not failures.empty else quality_report)
print(f'Votes linked to a bill: {len(linked_votes):,}/{len(fact_votes):,}')
display(coverage)
reading_not_stated = int(bridge_vote_bills['reading'].isna().sum())
print(f'Reading explicitly stated: {len(bridge_vote_bills) - reading_not_stated:,}/'
      f'{len(bridge_vote_bills):,}; otherwise reading=null means not stated in motion')
print('Explicit reading distribution:', bridge_vote_bills['reading'].dropna().value_counts().to_dict())
display(collection_summary)

outputs = {**tables, 'collection_summary': collection_summary, 'quality_report': quality_report}
for name, table in outputs.items():
    staged = OUTPUT_DIR / f'{name}.parquet.part'
    table.to_parquet(staged, index=False)
    staged.replace(OUTPUT_DIR / f'{name}.parquet')
print(f'Wrote {len(outputs)} tables to {OUTPUT_DIR.relative_to(ROOT)}')

if FULL_RUN:
    blocking = failures[failures['severity'].eq('blocking')]
    assert blocking.empty, f'blocking data-quality failures: {list(blocking["check"])}'

Quality: 26/26 checks pass


,check,violations,limit,severity,status
0,duplicate vote_id in fact_votes,0,0,blocking,pass
1,ballot vote_id absent from fact_votes,0,0,blocking,pass
2,counter vote_id absent from fact_votes,0,0,blocking,pass
3,secret-result vote_id absent from fact_votes,0,0,blocking,pass
4,duplicate member within one vote,0,0,blocking,pass
5,duplicate counter_id,0,0,blocking,pass
6,duplicate secret_result_id,0,0,blocking,pass
7,ballots in one vote above 120 seats,0,0,blocking,pass
8,bridge bill_id absent from dim_bills,0,0,blocking,pass
9,bridge vote_id absent from fact_votes,0,0,blocking,pass


Votes linked to a bill: 27,900/36,054


size   sum  linked_pct
knesset_num lu_item_type                        
16          2             1690  1690       100.0
            3              141     0         0.0
            4              781     0         0.0
            9             1155     0         0.0
17          2             3225  3225       100.0
            3              225     0         0.0
            4              621     0         0.0
            9              467     0         0.0
18          2             4053  4053       100.0
            3              275     0         0.0
            4              895     0         0.0
            9              380     0         0.0
            950              1     0         0.0
19          2             1863  1863       100.0
            3              160     0         0.0
            4              349     0         0.0
            9              128     0         0.0
20          2             6420  6420       100.0
            3              193     0         0.0
            4              736     0         0.0
            9              178     0         0.0
            11               1     0         0.0
            6000             1     0         0.0
            6003             1     0         0.0
21          2                7     7       100.0
            4                1     0         0.0
            9                8     0         0.0
22          2               27    27       100.0
            4               52     0         0.0
            9                7     0         0.0
23          2             1275  1275       100.0
            3               78     0         0.0
            4              170     0         0.0
            9               68     0         0.0
            6000            17     0         0.0
24          2             2578  2578       100.0
            3               59     0         0.0
            4              140     0         0.0
            9               56     0         0.0
            6000            49     0         0.0
25          2             6762  6762       100.0
            3              221     0         0.0
            4              375     0         0.0
            9               20     0         0.0
            6000           145     0         0.0

Reading explicitly stated: 15,217/27,900; otherwise reading=null means not stated in motion
Explicit reading distribution: {1.0: 7197, 2.0: 5165, 3.0: 2855}


,metric,count
0,full_run,1
1,vote_dates,2040
2,votes,36054
3,ballots,1948412
4,vote_counter_rows,67454
5,secret_vote_results,18
6,vote_bill_links,27900
7,bill_votes_reading_not_stated,12683
8,source_empty_electronic_votes,140
9,source_knesset_differs_from_vote,416


Wrote 11 tables to dataset/knesset_votes/prepared
